<a href="https://colab.research.google.com/github/Mogu-code/DataScience/blob/main/Nykaa_Campaign_Performance_FDS_DA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install and import libraries (As done in your FDS_sql notebook)
!pip install --upgrade duckdb pandas

In [15]:
import pandas as pd
import numpy as np
import duckdb

# Load the dataset
df = pd.read_csv('/nykaa_campaign_data.csv')

print("Dataset Loaded Successfully!")
df.head()

Dataset Loaded Successfully!


,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date
0,NY-CMP-1000,Social Media,College Students,21,"WhatsApp, YouTube",57804,6156,3616,2355,1867515,111.03,6.14,Hindi,20.98,College Students,29-04-2025
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18,YouTube,91801,3321,1971,1357,1046247,180.83,3.26,Hindi,7.24,College Students,06-04-2025
2,NY-CMP-1002,Influencer,Youth,23,"WhatsApp, Google, YouTube",15536,2182,952,755,197055,90.60,1.88,English,25.03,College Students,14-01-2025
3,NY-CMP-1003,Email,Working Women,18,"YouTube, Facebook, Instagram",88114,8413,2231,947,376906,249.07,0.60,Hindi,13.15,College Students,04-06-2025
4,NY-CMP-1004,Paid Ads,College Students,10,"Facebook, Instagram",96871,3743,2060,1258,518296,228.60,0.80,Hindi,7.29,Tier 2 City Customers,29-12-2024


In [16]:
# 1. Check for missing values
print("Missing values per column:\n", df.isnull().sum())



Missing values per column:
 Campaign_ID         0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Impressions         0
Clicks              0
Leads               0
Conversions         0
Revenue             0
Acquisition_Cost    0
ROI                 0
Language            0
Engagement_Score    0
Customer_Segment    0
Date                0
dtype: int64


In [17]:
# 2. Even if there are no NaNs, we demonstrate 'Imputation' logic
# Fill any potential missing Engagement Scores with the Median (Robust to outliers)
df['Engagement_Score'] = df['Engagement_Score'].fillna(df['Engagement_Score'].median())
df['Engagement_Score']


0        20.98
1         7.24
2        25.03
3        13.15
4         7.29
         ...  
55550    17.72
55551    25.26
55552    27.29
55553    10.14
55554    23.65
Name: Engagement_Score, Length: 55555, dtype: float64

In [18]:
# 3. Handle outliers in ROI using NumPy (Anomalies)
roi_mean = np.mean(df['ROI'])
roi_std = np.std(df['ROI'])
print(f"ROI Mean: {roi_mean:.2f}, ROI Std Dev: {roi_std:.2f}")

ROI Mean: 2.71, ROI Std Dev: 4.49


In [19]:
# Remove any data points that are extreme outliers (e.g., ROI > 50)
df = df[df['ROI'] < 50]
print("Data Cleaned and Outliers handled.")

Data Cleaned and Outliers handled.


In [20]:
# Query 1: Campaign Efficiency Ranking (Window Function - RANK)
# Rank campaigns within each type (Social Media, Email, etc.) based on ROI
q1 = """
SELECT Campaign_ID, Campaign_Type, ROI,
       RANK() OVER (PARTITION BY Campaign_Type ORDER BY ROI DESC) as Performance_Rank
FROM df
LIMIT 10;
"""
print("Query 1: Performance Ranking within Types")
display(duckdb.sql(q1).df())

Query 1: Performance Ranking within Types


,Campaign_ID,Campaign_Type,ROI,Performance_Rank
0,NY-CMP-44966,Paid Ads,49.98,1
1,NY-CMP-5942,Paid Ads,47.14,2
2,NY-CMP-8637,Paid Ads,45.57,3
3,NY-CMP-24494,Paid Ads,45.25,4
4,NY-CMP-7698,Paid Ads,42.52,5
5,NY-CMP-17837,Paid Ads,40.68,6
6,NY-CMP-14545,Paid Ads,40.16,7
7,NY-CMP-36425,Paid Ads,39.20,8
8,NY-CMP-31793,Paid Ads,38.04,9
9,NY-CMP-34917,Paid Ads,37.81,10


In [22]:
# Query 2: Cumulative Revenue Growth (Running Total)
q2 = """
SELECT Date, Revenue,
       SUM(Revenue) OVER (ORDER BY Date) as Cumulative_Revenue
FROM df
LIMIT 10;
"""
print("\nQuery 2: Running Total of Revenue")
display(duckdb.sql(q2).df())




Query 2: Running Total of Revenue


,Date,Revenue,Cumulative_Revenue
0,01-01-2025,150612,84828735.0
1,01-01-2025,174309,84828735.0
2,01-01-2025,238596,84828735.0
3,01-01-2025,12483,84828735.0
4,01-01-2025,401076,84828735.0
5,01-01-2025,194150,84828735.0
6,01-01-2025,108106,84828735.0
7,01-01-2025,1764294,84828735.0
8,01-01-2025,302918,84828735.0
9,01-01-2025,984520,84828735.0


In [23]:
# Query 3: Identifying High-Cost Outliers (Subquery)
# Find campaigns where cost is higher than the average cost of ALL campaigns
q3 = """
SELECT Campaign_ID, Campaign_Type, Acquisition_Cost
FROM df
WHERE Acquisition_Cost > (SELECT AVG(Acquisition_Cost) FROM df)
ORDER BY Acquisition_Cost DESC
LIMIT 5;
"""
print("\nQuery 3: Campaigns exceeding average cost")
display(duckdb.sql(q3).df())



Query 3: Campaigns exceeding average cost


,Campaign_ID,Campaign_Type,Acquisition_Cost
0,NY-CMP-34324,Paid Ads,15473.16
1,NY-CMP-16583,Paid Ads,12423.65
2,NY-CMP-52344,Paid Ads,10920.48
3,NY-CMP-3875,Influencer,9661.16
4,NY-CMP-1302,Influencer,9469.83


In [24]:

# Query 4: Language-Based ROI Performance (Group By + Having)
q4 = """
SELECT Language, ROUND(AVG(ROI), 2) as Avg_ROI
FROM df
GROUP BY Language
HAVING Avg_ROI > 2.0
ORDER BY Avg_ROI DESC;
"""
print("\nQuery 4: Best performing languages by ROI")
display(duckdb.sql(q4).df())


Query 4: Best performing languages by ROI


,Language,Avg_ROI
0,Hindi,2.74
1,Bengali,2.70
2,English,2.70
3,Tamil,2.67


In [25]:
# Query 5: Campaign "Classification" (Case Statement)
# Manually classifying campaigns into performance tiers
q5 = """
SELECT Campaign_ID, ROI,
       CASE
           WHEN ROI > 5 THEN 'Star Campaign'
           WHEN ROI BETWEEN 1 AND 5 THEN 'Average'
           ELSE 'Loss Making'
       END as Campaign_Tier
FROM df
LIMIT 10;
"""
print("\nQuery 5: Performance Classification")
display(duckdb.sql(q5).df())



Query 5: Performance Classification


,Campaign_ID,ROI,Campaign_Tier
0,NY-CMP-1000,6.14,Star Campaign
1,NY-CMP-1001,3.26,Average
2,NY-CMP-1002,1.88,Average
3,NY-CMP-1003,0.60,Loss Making
4,NY-CMP-1004,0.80,Loss Making
5,NY-CMP-1005,3.09,Average
6,NY-CMP-1006,1.17,Average
7,NY-CMP-1007,1.40,Average
8,NY-CMP-1008,3.73,Average
9,NY-CMP-1009,1.61,Average


In [26]:

# Query 6: Moving Average of Engagement (Smoothing Trends)
q6 = """
SELECT Date, Engagement_Score,
       AVG(Engagement_Score) OVER (ORDER BY Date ROWS BETWEEN 7 PRECEDING AND CURRENT ROW) as Weekly_Moving_Avg
FROM df
LIMIT 10;
"""
print("\nQuery 6: 7-Day Moving Average of Engagement")
display(duckdb.sql(q6).df())



Query 6: 7-Day Moving Average of Engagement


,Date,Engagement_Score,Weekly_Moving_Avg
0,01-01-2025,7.27,7.270000
1,01-01-2025,9.09,8.180000
2,01-01-2025,12.86,9.740000
3,01-01-2025,5.30,8.630000
4,01-01-2025,11.57,9.218000
5,01-01-2025,6.31,8.733333
6,01-01-2025,4.36,8.108571
7,01-01-2025,15.86,9.077500
8,01-01-2025,15.14,10.061250
9,01-01-2025,26.46,12.232500


In [27]:

# Query 7: Channel Saturation Analysis (Multi-level Grouping)
q7 = """
SELECT Campaign_Type, Target_Audience, COUNT(*) as Total_Campaigns, AVG(Clicks) as Avg_Clicks
FROM df
GROUP BY Campaign_Type, Target_Audience
ORDER BY Avg_Clicks DESC
LIMIT 5;
"""
print("\nQuery 7: Best Channel-Audience combinations")
display(duckdb.sql(q7).df())


Query 7: Best Channel-Audience combinations


,Campaign_Type,Target_Audience,Total_Campaigns,Avg_Clicks
0,Paid Ads,College Students,2203,4833.642306
1,Social Media,College Students,2237,4816.637461
2,Influencer,Premium Shoppers,2246,4810.318789
3,Influencer,College Students,2190,4775.260731
4,Email,Premium Shoppers,2243,4770.807847


In [28]:

# Query 8: "Churning" Channels (Low Conversion Rate Analysis)
# Find channels where the Lead-to-Conversion ratio is dangerously low (< 10%)
q8 = """
SELECT Campaign_Type,
       AVG(Conversions * 100.0 / NULLIF(Leads, 0)) as Conversion_Rate
FROM df
GROUP BY Campaign_Type
HAVING Conversion_Rate < 60
ORDER BY Conversion_Rate ASC;
"""
print("\nQuery 8: Low performing 'Churn' risk channels")
display(duckdb.sql(q8).df())



Query 8: Low performing 'Churn' risk channels


,Campaign_Type,Conversion_Rate
0,Social Media,54.879619
1,Paid Ads,54.904195
2,Influencer,55.014689
3,SEO,55.061524
4,Email,55.179089


In [29]:

# Query 9: Proxy Sentiment Analysis (Logic based on Engagement & ROI)
q9 = """
SELECT Campaign_ID, Customer_Segment,
       (Engagement_Score * 0.6 + ROI * 0.4) as Brand_Sentiment_Index
FROM df
ORDER BY Brand_Sentiment_Index DESC
LIMIT 5;
"""
print("\nQuery 9: Derived Sentiment Index")
display(duckdb.sql(q9).df())


Query 9: Derived Sentiment Index


,Campaign_ID,Customer_Segment,Brand_Sentiment_Index
0,NY-CMP-1227,College Students,35.558
1,NY-CMP-44966,Working Women,35.472
2,NY-CMP-3383,Youth,35.204
3,NY-CMP-8168,Tier 2 City Customers,34.318
4,NY-CMP-55921,Tier 2 City Customers,34.170


In [30]:

# Query 10: Resource Optimization (Cost per Conversion)
q10 = """
SELECT Language, Target_Audience,
       SUM(Acquisition_Cost) / SUM(Conversions) as Cost_Per_Conversion
FROM df
GROUP BY Language, Target_Audience
ORDER BY Cost_Per_Conversion ASC
LIMIT 5;
"""
print("\nQuery 10: Most cost-efficient segments")
display(duckdb.sql(q10).df())


Query 10: Most cost-efficient segments


,Language,Target_Audience,Cost_Per_Conversion
0,Tamil,College Students,0.348225
1,English,College Students,0.350633
2,Hindi,College Students,0.352279
3,Tamil,Premium Shoppers,0.353278
4,Bengali,Tier 2 City Customers,0.355201
